# GTFS school-day filter (DuckDB)

This notebook reads per-feed Parquet outputs and applies a Weekday school-day filter.
It also shows which routes have both `SDon` and non-`SDon` service IDs.


In [1]:
import duckdb
from pathlib import Path

PARQ_BASE = Path('parquet')  # change if your parquet lives elsewhere
con = duckdb.connect()

def create_union_view(table_name: str, cast_cols: dict[str, str] | None = None):
    cast_cols = cast_cols or {}
    folder = PARQ_BASE / table_name
    if not folder.exists():
        raise FileNotFoundError(f'No parquet folder found at {folder}')
    path = (folder / '*.parquet').as_posix()
    base = f"read_parquet('{path}', union_by_name=True)"
    if cast_cols:
        cols = ', '.join(cast_cols.keys())
        casts = ', '.join([f"CAST({col} AS {typ}) AS {col}" for col, typ in cast_cols.items()])
        sql = f"SELECT * EXCLUDE ({cols}), {casts} FROM {base}"
    else:
        sql = f"SELECT * FROM {base}"
    con.execute(f"CREATE OR REPLACE TEMP VIEW {table_name} AS {sql}")

# Register tables; cast service_id to VARCHAR to avoid mixed-type errors
create_union_view('dim_trips', {'service_id': 'VARCHAR'})
create_union_view('dim_routes')
create_union_view('calendar_base', {'service_id': 'VARCHAR'})
create_union_view('dim_stops')
create_union_view('fact_stop_events')

In [2]:
# Parameters
day_type = 'Weekday'  # Weekday | Saturday | Sunday
school_choice = 'School day only'  # All | School day only | Non-school day only
selected_feeds = []  # e.g. ['miami-dade']; empty means all


In [3]:
# Build feed filter snippets
sel = list(selected_feeds or [])
if sel:
    values = ','.join(['(?)'] * len(sel))
    chosen_cte = f"chosen_feeds(feed_id) AS (VALUES {values}),"
    feed_pred = 'feed_id IN (SELECT feed_id FROM chosen_feeds)'
else:
    chosen_cte = ''
    feed_pred = 'TRUE'


In [4]:
routes = con.execute('SELECT * FROM dim_routes').fetchdf()
trips = con.execute('SELECT * FROM dim_trips').fetchdf()
stops = con.execute('SELECT * FROM dim_stops').fetchdf()
facts = con.execute('SELECT * FROM fact_stop_events').fetchdf()
facts

,route_id,direction_id,service_id,stop_id,stop_sequence,arrival_sec,trip_id,feed_id
0,14456,0,11,795,1,19920,4828771,miami-dade
1,14456,0,11,796,2,19980,4828771,miami-dade
2,14456,0,11,797,3,20040,4828771,miami-dade
3,14456,0,11,798,4,20100,4828771,miami-dade
4,14456,0,11,799,5,20160,4828771,miami-dade
...,...,...,...,...,...,...,...,...
8397803,SIM9,1,YU_D5-Weekday-SDon,201136,26,61159,YU_D5-Weekday-SDon-094500_MISC_747,mta-nyct-bus-si
8397804,SIM9,1,YU_D5-Weekday-SDon,201139,27,61257,YU_D5-Weekday-SDon-094500_MISC_747,mta-nyct-bus-si
8397805,SIM9,1,YU_D5-Weekday-SDon,201141,28,61324,YU_D5-Weekday-SDon-094500_MISC_747,mta-nyct-bus-si
8397806,SIM9,1,YU_D5-Weekday-SDon,200368,29,61421,YU_D5-Weekday-SDon-094500_MISC_747,mta-nyct-bus-si


In [5]:
import pandas as pd
import plotly.express as px

In [6]:
facts[(facts['service_id'].isin(['YU_D5-Weekday','YU_S5-Weekday']))]

,route_id,direction_id,service_id,stop_id,stop_sequence,arrival_sec,trip_id,feed_id
7709929,SIM15,0,YU_S5-Weekday,200386,1,17940,YU_S5-Weekday-029900_MISC_723,mta-nyct-bus-si
7709930,SIM15,0,YU_S5-Weekday,202099,2,18043,YU_S5-Weekday-029900_MISC_723,mta-nyct-bus-si
7709931,SIM15,0,YU_S5-Weekday,200496,3,18125,YU_S5-Weekday-029900_MISC_723,mta-nyct-bus-si
7709932,SIM15,0,YU_S5-Weekday,202724,4,18177,YU_S5-Weekday-029900_MISC_723,mta-nyct-bus-si
7709933,SIM15,0,YU_S5-Weekday,203113,5,18211,YU_S5-Weekday-029900_MISC_723,mta-nyct-bus-si
...,...,...,...,...,...,...,...,...
8114149,SIM9,1,YU_D5-Weekday,201136,26,61159,YU_D5-Weekday-094500_MISC_747,mta-nyct-bus-si
8114150,SIM9,1,YU_D5-Weekday,201139,27,61257,YU_D5-Weekday-094500_MISC_747,mta-nyct-bus-si
8114151,SIM9,1,YU_D5-Weekday,201141,28,61324,YU_D5-Weekday-094500_MISC_747,mta-nyct-bus-si
8114152,SIM9,1,YU_D5-Weekday,200368,29,61421,YU_D5-Weekday-094500_MISC_747,mta-nyct-bus-si


In [10]:
temp[temp['route_id'] == 'SIM1'].sort_values('arrival_sec')

,route_id,direction_id,service_id,stop_id,stop_sequence,arrival_sec,trip_id,feed_id
8096582,SIM1,1,YU_D5-Weekday,200368,31,57174,YU_D5-Weekday-089000_MISC_735,mta-nyct-bus-si
7713288,SIM1,1,YU_S5-Weekday,200368,31,57774,YU_S5-Weekday-090000_MISC_792,mta-nyct-bus-si
8098067,SIM1,1,YU_D5-Weekday,200368,31,57774,YU_D5-Weekday-090000_SIM1_557,mta-nyct-bus-si
7712059,SIM1,1,YU_S5-Weekday,200368,31,58674,YU_S5-Weekday-091500_MISC_778,mta-nyct-bus-si
8095342,SIM1,1,YU_D5-Weekday,200368,31,58854,YU_D5-Weekday-091800_MISC_749,mta-nyct-bus-si
8095379,SIM1,1,YU_D5-Weekday,200368,31,59362,YU_D5-Weekday-092500_MISC_737,mta-nyct-bus-si
7712096,SIM1,1,YU_S5-Weekday,200368,31,59662,YU_S5-Weekday-093000_MISC_771,mta-nyct-bus-si
8097960,SIM1,1,YU_D5-Weekday,200368,31,59782,YU_D5-Weekday-093200_SIM1_559,mta-nyct-bus-si
8096915,SIM1,1,YU_D5-Weekday,200368,31,60202,YU_D5-Weekday-093900_SIM10_599,mta-nyct-bus-si
7712170,SIM1,1,YU_S5-Weekday,200368,31,60622,YU_S5-Weekday-094500_MISC_801,mta-nyct-bus-si


In [25]:
facts[facts['service_id'].str.contains('OF')].drop_duplicates('service_id')

,route_id,direction_id,service_id,stop_id,stop_sequence,arrival_sec,trip_id,feed_id
5743398,M125,0,OF_O5-Weekday,404362,1,3600,OF_O5-Weekday-006000_M125_501,mta-nyct-bus-manhattan
6256541,M125,0,OF_D5-Weekday,404362,1,3600,OF_D5-Weekday-006000_M125_501,mta-nyct-bus-manhattan
6295526,M125,1,OF_D5-Saturday,101066,1,1800,OF_D5-Saturday-003000_M125_501,mta-nyct-bus-manhattan
6330375,M125,0,OF_D5-Sunday,404362,1,3600,OF_D5-Sunday-006000_M125_501,mta-nyct-bus-manhattan
6680584,M125,0,OF_D5-Weekday-SDon,404362,1,3600,OF_D5-Weekday-SDon-006000_M125_501,mta-nyct-bus-manhattan


In [26]:
temp = facts[(facts['service_id'].isin(['OF_O5-Weekday','OF_D5-Weekday','OF_D5-Weekday-SDon'])) & (facts['stop_id'] == 404362)]


px.histogram(temp['arrival_sec'],color=temp['service_id'], nbins=24, barmode='overlay')

In [57]:
# identify routes that actually vary
trips['is_sdon'] = trips.service_id.str.contains("SDon")
routes_with_school_split = (
    trips.groupby("route_id")["is_sdon"]
    .any()
)
routes_with_school_split

# attach back to trips
trips = trips.merge(
    routes_with_school_split.rename("has_school_split"),
    on="route_id"
)
trips

,trip_id,route_id,direction_id,trip_headsign,feed_id,service_id,is_sdon,has_school_split
0,4828771,14456,0,DOWNTOWN,miami-dade,11,False,False
1,4828981,14456,0,DOWNTOWN,miami-dade,11,False,False
2,4828772,14456,0,DOWNTOWN,miami-dade,11,False,False
3,4828982,14456,0,DOWNTOWN,miami-dade,11,False,False
4,4828773,14456,0,DOWNTOWN,miami-dade,11,False,False
...,...,...,...,...,...,...,...,...
255111,YU_D5-Weekday-SDon-088500_SIM9_688,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday-SDon,True,True
255112,YU_D5-Weekday-SDon-104500_SIM9_688,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday-SDon,True,True
255113,YU_D5-Weekday-SDon-091500_SIM9_691,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday-SDon,True,True
255114,YU_D5-Weekday-SDon-107500_SIM9_691,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday-SDon,True,True


In [64]:
trips[~trips.has_school_split]['route_id'].unique()

array(['14456', '14457', '14458', '23889', '23892', '23896', '23903',
       '23904', '23905', '23906', '28803', '28804', '28805', '28806',
       '28807', '28812', '28827', '29124', '23885', '23886', '23888',
       '23891', '23893', '23894', '23895', '23907', '23910', '23915',
       '23920', '28809', '28810', '28817', '28818', '28819', '28820',
       '28822', '28823', '28826', '23908', '23909', '23912', '23914',
       '23916', '23917', '23918', '23919', '23924', '23929', '28802',
       '28808', '28811', '28816', '28824', '30315', '30631', '30632',
       '30633', '30634', '30635', '30636', '30637', '30638', '30639',
       '30640', '30641', '30642', '30643', '30644', '30645', '30646',
       '30647', '30648', '30649', '30650', '30651', '30652', '30653',
       '30654', '30655', '30656', '30657', '30658', '30659', '30660',
       '30661', '30662', '30663', '30664', '30665', '30666', '30667',
       '30668', '30669', '30670', '30671', '30672', '30673', '30674',
       '30675', '306

,trip_id,route_id,direction_id,trip_headsign,feed_id,service_id,is_sdon,has_school_split
240828,YU_S5-Weekday-029900_MISC_723,SIM15,0,DOWNTOWN LOOP via CHURCH ST via WATER ST,mta-nyct-bus-si,YU_S5-Weekday,False,True
240829,YU_S5-Weekday-034900_MISC_752,SIM15,0,DOWNTOWN LOOP via CHURCH ST via WATER ST,mta-nyct-bus-si,YU_S5-Weekday,False,True
240830,YU_S5-Weekday-036900_MISC_767,SIM15,0,DOWNTOWN LOOP via CHURCH ST via WATER ST,mta-nyct-bus-si,YU_S5-Weekday,False,True
240831,YU_S5-Weekday-040500_MISC_797,SIM15,0,DOWNTOWN LOOP via CHURCH ST via WATER ST,mta-nyct-bus-si,YU_S5-Weekday,False,True
240832,YU_S5-Weekday-045000_MISC_735,SIM15,0,DOWNTOWN LOOP via CHURCH ST via WATER ST,mta-nyct-bus-si,YU_S5-Weekday,False,True
...,...,...,...,...,...,...,...,...
248927,YU_D5-Weekday-088500_SIM9_688,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday,False,True
248928,YU_D5-Weekday-104500_SIM9_688,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday,False,True
248929,YU_D5-Weekday-091500_SIM9_691,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday,False,True
248930,YU_D5-Weekday-107500_SIM9_691,SIM9,1,ELTINGVILLE via F CAP via HYLAN BL,mta-nyct-bus-si,YU_D5-Weekday,False,True


In [69]:
school_day_trips[school_day_trips['feed_id'] !='miami-dade'].drop_duplicates(['route_id','service_id'],keep=False).sort_values('route_id')

,trip_id,route_id,direction_id,trip_headsign,feed_id,service_id,has_school_split,is_sdon
122265,UP_D5-Weekday-SDon-084500_X2737_724,B4,1,BAY RIDGE NARROWS AV,mta-nyct-bus-brooklyn,UP_D5-Weekday-SDon,True,True
121366,JG_D5-Weekday-SDon-BM-142000_B61_801,B61,0,DOWNTOWN BKLYN FULTON MALL via RED HOOK,mta-nyct-bus-brooklyn,JG_D5-Weekday-SDon-BM,True,True
121652,JG_D5-Weekday-SDon-BM-141800_B8_101,B8,0,BROWNSVILLE ROCKAWAY AV,mta-nyct-bus-brooklyn,JG_D5-Weekday-SDon-BM,True,True
59204,GH_D5-Weekday-SDon-BM-143500_BX41_901,BX41,1,THE HUB 150 ST via WEBSTER,mta-nyct-bus-bronx,GH_D5-Weekday-SDon-BM,True,True
206667,OH_D5-Weekday-SDon-BM-143300_M101_1,M103,1,CITY HALL via LEX AV,mta-nyct-bus-manhattan,OH_D5-Weekday-SDon-BM,True,True
206197,OH_D5-Weekday-SDon-BM-141800_M15_201,M15,1,SOUTH FERRY via 2 AV,mta-nyct-bus-manhattan,OH_D5-Weekday-SDon-BM,True,True
237256,QV_D5-Weekday-SDon-BM-142500_Q2_51,Q2,1,RUSH JAMAICA BUS TERMINAL,mta-nyct-bus-queens,QV_D5-Weekday-SDon-BM,True,True
252659,CA_D5-Weekday-SDon-087100_MISC_357,S44,0,ST GEORGE FERRY,mta-nyct-bus-si,CA_D5-Weekday-SDon,True,True
251201,CA_D5-Weekday-SDon-043500_S7686_147,S55,1,HUGUENOT LUTEN AV,mta-nyct-bus-si,CA_D5-Weekday-SDon,True,True
252013,CA_D5-Weekday-SDon-043500_MISC_280,S56,1,HUGUENOT LUTEN AV,mta-nyct-bus-si,CA_D5-Weekday-SDon,True,True


In [ ]:
# school-day view
school_day_trips = trips[
    (trips.service_id.str.contains("SDon")) |
    (~trips.has_school_split)
]

# non-school-day view
non_school_day_trips = trips[
    (~trips.service_id.str.contains("SDon")) |
    (~trips.has_school_split)
]

In [ ]:
# Identify routes that have both SDon and non-SDon weekday service_ids
sql_routes_with_both = f"""
WITH
{chosen_cte}
weekday_svcs AS (
  SELECT DISTINCT feed_id, service_id
  FROM calendar_base
  WHERE {feed_pred}
    AND (monday=1 OR tuesday=1 OR wednesday=1 OR thursday=1 OR friday=1)
),
route_services AS (
  SELECT DISTINCT t.feed_id, t.route_id, t.service_id
  FROM dim_trips t
  JOIN weekday_svcs w
    ON t.feed_id = w.feed_id AND t.service_id = w.service_id
  WHERE {feed_pred}
),
routes_with_sdon AS (
  SELECT DISTINCT feed_id, route_id
  FROM route_services
  WHERE CAST(service_id AS VARCHAR) LIKE '%SDon'
),
routes_with_non AS (
  SELECT DISTINCT feed_id, route_id
  FROM route_services
  WHERE CAST(service_id AS VARCHAR) NOT LIKE '%SDon'
),
routes_with_both AS (
  SELECT s.feed_id, s.route_id
  FROM routes_with_sdon s
  INNER JOIN routes_with_non n
    ON s.feed_id = n.feed_id AND s.route_id = n.route_id
)
SELECT *
FROM routes_with_both
ORDER BY feed_id, route_id
"""

params = []
if sel:
    params += sel

routes_with_both_df = con.execute(sql_routes_with_both, params).fetchdf()
routes_with_both_df

,feed_id,route_id
0,mta-nyct-bus-bronx,BX1
1,mta-nyct-bus-bronx,BX10
2,mta-nyct-bus-bronx,BX11
3,mta-nyct-bus-bronx,BX12
4,mta-nyct-bus-bronx,BX12+
...,...,...
326,mta-nyct-bus-si,SIM5
327,mta-nyct-bus-si,SIM6
328,mta-nyct-bus-si,SIM7
329,mta-nyct-bus-si,SIM8


In [7]:
# Core filter: school-day only (change predicate for non-school day only)
sql_filtered_services = f"""
WITH
{chosen_cte}
weekday_svcs AS (
  SELECT DISTINCT feed_id, service_id
  FROM calendar_base
  WHERE {feed_pred}
    AND (monday=1 OR tuesday=1 OR wednesday=1 OR thursday=1 OR friday=1)
),
route_services AS (
  SELECT DISTINCT t.feed_id, t.route_id, t.service_id
  FROM dim_trips t
  JOIN weekday_svcs w
    ON t.feed_id = w.feed_id AND t.service_id = w.service_id
  WHERE {feed_pred}
),
routes_with_sdon AS (
  SELECT DISTINCT feed_id, route_id
  FROM route_services
  WHERE CAST(service_id AS VARCHAR) LIKE '%SDon'
),
routes_with_non AS (
  SELECT DISTINCT feed_id, route_id
  FROM route_services
  WHERE CAST(service_id AS VARCHAR) NOT LIKE '%SDon'
),
routes_with_both AS (
  SELECT s.feed_id, s.route_id
  FROM routes_with_sdon s
  INNER JOIN routes_with_non n
    ON s.feed_id = n.feed_id AND s.route_id = n.route_id
),
filtered_services AS (
  SELECT rs.*
  FROM route_services rs
  LEFT JOIN routes_with_both b
    ON rs.feed_id = b.feed_id AND rs.route_id = b.route_id
  WHERE
    (b.route_id IS NOT NULL AND CAST(rs.service_id AS VARCHAR) LIKE '%SDon')
    OR (b.route_id IS NULL)
)
SELECT *
FROM filtered_services
ORDER BY feed_id, route_id, service_id
"""

params = []
if sel:
    params += sel

filtered_services_df = con.execute(sql_filtered_services, params).fetchdf()
filtered_services_df.head(50)


,feed_id,route_id,service_id
0,miami-dade,14456,11
1,miami-dade,14457,11
2,miami-dade,14458,11
3,miami-dade,23885,21
4,miami-dade,23886,21
5,miami-dade,23888,21
6,miami-dade,23889,21
7,miami-dade,23891,21
8,miami-dade,23892,21
9,miami-dade,23893,21


## Switching filter behavior
- **School day only** (current): `LIKE '%SDon'`
- **Non-school day only**: change to `NOT LIKE '%SDon'`
- **All**: skip `filtered_services` and use `weekday_svcs` instead


In [ ]:

filtered_services_df = con.execute(sql_filtered_services, params).fetchdf()
filtered_services_df.head(50)

In [ ]:
# Quick samples
con.execute('SELECT feed_id, route_id, service_id FROM dim_trips LIMIT 20').fetchdf()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2192496457.py, line 3)

In [13]:
q2 = """
WITH weekday_svcs AS (
  SELECT DISTINCT feed_id, service_id
  FROM calendar_base
  WHERE (monday=1 OR tuesday=1 OR wednesday=1 OR thursday=1 OR friday=1)
),
route_services AS (
  SELECT DISTINCT t.feed_id, t.route_id, t.service_id
  FROM dim_trips t
  JOIN weekday_svcs w
    ON t.feed_id = w.feed_id AND t.service_id = w.service_id
),
routes_with_sdon AS (
  SELECT DISTINCT feed_id, route_id
  FROM route_services
  WHERE CAST(service_id AS VARCHAR) LIKE '%SDon'
)
SELECT DISTINCT rs.feed_id, rs.route_id
FROM route_services rs
LEFT JOIN routes_with_sdon s
  ON rs.feed_id = s.feed_id AND rs.route_id = s.route_id
WHERE s.route_id IS NULL
ORDER BY rs.feed_id, rs.route_id
"""

df_no_sdon = con.execute(q2).fetchdf()
df_no_sdon


,feed_id,route_id
0,miami-dade,14456
1,miami-dade,14457
2,miami-dade,14458
3,miami-dade,23885
4,miami-dade,23886
...,...,...
126,mta-nyct-bus-busco,BXM11
127,mta-nyct-bus-busco,BXM18
128,mta-nyct-bus-busco,BXM2
129,mta-nyct-bus-busco,BXM3


In [14]:
# Specific route_ids you choose
route_ids = ['BX1', 'SIM5', 'BX12','BXM4']  # replace with your route_ids

q1 = """
SELECT DISTINCT feed_id, route_id, service_id
FROM dim_trips
WHERE route_id IN ({})
ORDER BY feed_id, route_id, service_id
""".format(",".join([f"'{r}'" for r in route_ids]))

df_routes = con.execute(q1).fetchdf()
df_routes


,feed_id,route_id,service_id
0,mta-nyct-bus-bronx,BX1,KB_D5-Saturday
1,mta-nyct-bus-bronx,BX1,KB_D5-Sunday
2,mta-nyct-bus-bronx,BX1,KB_D5-Weekday
3,mta-nyct-bus-bronx,BX1,KB_D5-Weekday-SDon
4,mta-nyct-bus-bronx,BX1,KB_O5-Weekday
5,mta-nyct-bus-bronx,BX12,GH_D5-Saturday
6,mta-nyct-bus-bronx,BX12,GH_D5-Sunday
7,mta-nyct-bus-bronx,BX12,GH_D5-Weekday
8,mta-nyct-bus-bronx,BX12,GH_D5-Weekday-SDon
9,mta-nyct-bus-bronx,BX12,GH_O5-Weekday
